# 13 — Beyond citations: novelty, disruption and other new measures (Book 2, Chapter 13)

Runs the chapter's analyses on the malaria vaccine corpus: AI-style vocabulary in abstracts, iCite translation scores, the CD index with and without old references, first-time journal pairs, and Rao-Stirling diversity. Every number in the chapter comes from this notebook, run on 17 November 2026.

In [1]:
# Settings
import os, subprocess, sys, pandas as pd
RUN_FETCH = False   # True re-downloads from OpenAlex (about 4,500 requests); needs OPENALEX_API_KEY in the environment
os.environ.setdefault("RAW_DIR", "../data/raw")
assert not RUN_FETCH or os.environ.get("OPENALEX_API_KEY"), "Set OPENALEX_API_KEY as an environment variable, never in a file"
pd.set_option("display.width", 200)

## 1. Fetch from OpenAlex (optional)

Reference lists and author records for the corpus, reference metadata, and the three citation counts per sampled paper for the CD index. The script resumes where it stopped and never prints the key.

In [2]:
if RUN_FETCH:
    for step in ["corpus", "refmeta", "allrefs", "cd"]:
        subprocess.run([sys.executable, "../src/13_fetch_openalex.py", step], check=True)

## 2. AI-style vocabulary and translation scores

In [3]:
subprocess.run([sys.executable, "../src/13a_ai_vocabulary_apt.py"], check=True)
pd.read_csv("../data/checks/ai_vocabulary_by_year.csv").query("year >= 2019").round(3)

 year  abstracts  observed  expected   gap  gap_lo  gap_hi
 2010        385     0.301       NaN   NaN     NaN     NaN
 2011        373     0.276       NaN   NaN     NaN     NaN
 2012        351     0.308       NaN   NaN     NaN     NaN
 2013        392     0.276       NaN   NaN     NaN     NaN
 2014        379     0.330       NaN   NaN     NaN     NaN
 2015        412     0.337       NaN   NaN     NaN     NaN
 2016        392     0.347       NaN   NaN     NaN     NaN
 2017        419     0.308       NaN   NaN     NaN     NaN
 2018        368     0.361       NaN   NaN     NaN     NaN
 2019        384     0.333       NaN   NaN     NaN     NaN
 2020        415     0.359       NaN   NaN     NaN     NaN
 2021        451     0.359       NaN   NaN     NaN     NaN
 2022        482     0.332       NaN   NaN     NaN     NaN
 2023        403     0.402     0.332 0.070  -0.004   0.136
 2024        418     0.517     0.332 0.185   0.081   0.249
 2025        470     0.589     0.332 0.257   0.121   0.3

,year,abstracts,observed,expected,gap,gap_lo,gap_hi
9,2019,384,0.333,NaN,NaN,NaN,NaN
10,2020,415,0.359,NaN,NaN,NaN,NaN
11,2021,451,0.359,NaN,NaN,NaN,NaN
12,2022,482,0.332,NaN,NaN,NaN,NaN
13,2023,403,0.402,0.332,0.070,-0.004,0.136
14,2024,418,0.517,0.332,0.185,0.081,0.249
15,2025,470,0.589,0.332,0.257,0.121,0.315
16,2026,414,0.614,0.332,0.282,0.107,0.343


## 3. Disruption (CD index), with and without references before 1990

In [4]:
subprocess.run([sys.executable, "../src/13b_disruption.py"], check=True)
r = pd.read_csv("../data/checks/disruption_by_period.csv")
r[["period", "n", "share_refs_pre1990", "cd_all", "cdnok_all", "cdnok_all_lo", "cdnok_all_hi", "cdnok_1990", "cdnok_1990_lo", "cdnok_1990_hi"]].round(3)

 period   n  uncited_in_window  share_refs_pre1990  cd_all  cd_all_lo  cd_all_hi  cd_1990  cd_1990_lo  cd_1990_hi  cdnok_all  cdnok_all_lo  cdnok_all_hi  cdnok_1990  cdnok_1990_lo  cdnok_1990_hi  n_cited5  cdnok_all_cited5
1990-94  72                  2              0.6942 -0.0082    -0.0123    -0.0047   0.0337      0.0004      0.0761    -0.4524       -0.5675       -0.3309      0.0018        -0.1336         0.1353        58           -0.5517
1995-99  72                  3              0.3393 -0.0031    -0.0089     0.0053  -0.0079     -0.0153      0.0034    -0.4952       -0.5908       -0.3842     -0.4150        -0.5097        -0.3060        60           -0.5306
2000-04  72                  2              0.1716 -0.0096    -0.0140    -0.0057  -0.0099     -0.0147     -0.0061    -0.4871       -0.5791       -0.3915     -0.4555        -0.5471        -0.3581        59           -0.5102
2005-09  72                  1              0.1069 -0.0066    -0.0093    -0.0042  -0.0088     -0.0123     -0

,period,n,share_refs_pre1990,cd_all,cdnok_all,cdnok_all_lo,cdnok_all_hi,cdnok_1990,cdnok_1990_lo,cdnok_1990_hi
0,1990-94,72,0.694,-0.008,-0.452,-0.568,-0.331,0.002,-0.134,0.135
1,1995-99,72,0.339,-0.003,-0.495,-0.591,-0.384,-0.415,-0.510,-0.306
2,2000-04,72,0.172,-0.010,-0.487,-0.579,-0.392,-0.455,-0.547,-0.358
3,2005-09,72,0.107,-0.007,-0.579,-0.667,-0.488,-0.564,-0.655,-0.462
4,2010-14,72,0.068,-0.003,-0.557,-0.644,-0.452,-0.538,-0.627,-0.440
5,2015-20,144,0.067,-0.003,-0.424,-0.495,-0.346,-0.421,-0.492,-0.351


**Hand check.** One paper recomputed from its three counts: n_i = A − B, n_j = B, n_k = C − B − 1 (the paper itself cites its references).

In [5]:
d = pd.read_csv("../data/checks/disruption_sample.csv").set_index("id")
e = d.loc["W2031565907"]
ni, nj, nk = e.A - e.B_all, e.B_all, e.C_all - e.B_all - 1
print(dict(n_i=ni, n_j=nj, n_k=nk), "CD =", round((ni - nj) / (ni + nj + nk), 4), "ratio =", (ni - nj) / (ni + nj))
assert abs((ni - nj) / (ni + nj + nk) - e.cd_all) < 1e-9

{'n_i': np.int64(0), 'n_j': np.int64(8), 'n_k': np.int64(854)} CD = -0.0093 ratio = -1.0


## 4. Novelty: first-time journal pairs

In [6]:
subprocess.run([sys.executable, "../src/13d_new_pairs.py"], check=True)
pd.read_csv("../data/checks/new_pairs_by_period.csv").round(3)

            n  new_pairs_all  new_pairs_10y  any_new_all
period                                                  
1970s      74          0.345          0.348        0.824
1980s     305          0.240          0.243        0.885
1990-94   372          0.158          0.163        0.828
1995-99   456          0.154          0.161        0.822
2000-04   854          0.162          0.171        0.884
2005-09  1317          0.154          0.159        0.868
2010-14  1736          0.138          0.142        0.881
2015-19  1865          0.138          0.143        0.905
2020-26  2787          0.176          0.185        0.919
corpus papers per decade: {1910: 1, 1920: 1, 1930: 1, 1940: 13, 1950: 3, 1960: 19, 1970: 135, 1980: 547, 1990: 1356, 2000: 2499, 2010: 3830, 2020: 2962}


,period,n,new_pairs_all,new_pairs_10y,any_new_all
0,1970s,74,0.345,0.348,0.824
1,1980s,305,0.240,0.243,0.885
2,1990-94,372,0.158,0.163,0.828
3,1995-99,456,0.154,0.161,0.822
4,2000-04,854,0.162,0.171,0.884
5,2005-09,1317,0.154,0.159,0.868
6,2010-14,1736,0.138,0.142,0.881
7,2015-19,1865,0.138,0.143,0.905
8,2020-26,2787,0.176,0.185,0.919


## 5. Within-sample novelty (trap T39) and Rao-Stirling diversity

In [7]:
subprocess.run([sys.executable, "../src/13c_novelty_diversity.py"], check=True)
pd.read_csv("../data/checks/novelty_diversity_by_period.csv").round(3)

sources 3551 pairs 199406 subfields 201
           n  refs_found  novel_tail  z_median  rao_stirling  variety
period                                                               
1990-94  292       0.988       0.966     0.621         0.324    4.538
1995-99  292       0.989       0.949     0.684         0.307    4.774
2000-04  292       0.989       0.959     0.850         0.329    4.822
2005-09  292       0.989       0.966     0.782         0.296    4.757
2010-14  292       0.988       0.969     0.815         0.311    5.216
2015-19  292       0.988       0.973     0.850         0.352    5.832
2020-26  512       0.990       0.955     0.900         0.388    6.090
                      n  novel_tail  rao_stirling
first_region                                     
elsewhere           881       0.968         0.356
sub-Saharan Africa  173       0.954         0.369
unknown              42       0.905         0.349


,period,n,refs_found,novel_tail,z_median,rao_stirling,variety
0,1990-94,292,0.988,0.966,0.621,0.324,4.538
1,1995-99,292,0.989,0.949,0.684,0.307,4.774
2,2000-04,292,0.989,0.959,0.850,0.329,4.822
3,2005-09,292,0.989,0.966,0.782,0.296,4.757
4,2010-14,292,0.988,0.969,0.815,0.311,5.216
5,2015-19,292,0.988,0.973,0.850,0.352,5.832
6,2020-26,512,0.990,0.955,0.900,0.388,6.090
